# Solar Filament Segmentation 2026 — U-Net + Connected Components

**Pipeline**
1. **U-Net** (`segmentation_models_pytorch`, ImageNet-pretrained EfficientNet-B3 encoder, 1-channel input) predicts a per-pixel filament probability. It is trained on all labelled images at full resolution (random 1024×1024 crops of the 2048×2048 originals).
2. Inference on the whole 2048×2048 image with flip test-time augmentation.
3. **Post-processing** turns the probability map into individual filaments: threshold → smooth outline & fill holes → join nearby fragments → drop small / low-confidence blobs.
4. Write `submission.csv` (pycocotools RLE counts).

**How to run**
* `TRAIN = False` (default): loads the released weights `weights/unet_effb3.pth` and reproduces `submission.csv` without training.
* `TRAIN = True`: trains from scratch (~2 h on a single T4 GPU) and overwrites the weights.
* Data: on Kaggle the competition data is found automatically under `/kaggle/input`; locally, unzip `MAGFiLO_1.0_Kaggle_2026` into `data/`.

In [ ]:
# Kaggle already has torch/numpy/opencv; locally use `pip install -r requirements.txt` instead.
!pip install -q segmentation-models-pytorch albumentations pycocotools scikit-image

In [ ]:
import json, random
from collections import defaultdict
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import albumentations as A
import segmentation_models_pytorch as smp
from albumentations.pytorch import ToTensorV2
from pycocotools import mask as mask_utils
from scipy import ndimage
from skimage.measure import regionprops
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

TRAIN = False                             # False: load released weights; True: train from scratch
WEIGHTS = Path("weights/unet_effb3.pth")

CFG = dict(
    encoder="efficientnet-b3",
    crop_size=1024,     # train on full-resolution crops; predict on the whole 2048 image
    batch_size=4,       # per GPU; lower to 2 if the GPU runs out of memory
    epochs=30,
    lr=3e-4,
    num_workers=2,
    seed=42,
)
# Post-processing in 2048px units, tuned on a held-out split in earlier experiments.
POST = dict(thr=0.4, merge_r=12, min_area=400, close_r=4, min_conf=0.85)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
seed_everything(CFG["seed"])

# Locate the dataset: Kaggle mount first, then a local `data/` folder.
ANN_NAME = "MAGFiLO_1.0_Annotations_kaggle2026_train.json"
ANN_PATH = next(p for root in [Path("/kaggle/input"), Path("data")] if root.exists() for p in root.rglob(ANN_NAME))
BASE = ANN_PATH.parent.parent
TRAIN_IMG_DIR = BASE / "train/train_images"
TEST_IMG_DIR = BASE / "test/test_images"
print(DEVICE, BASE)

## 1. Training data
Each entry in `images` is one annotator's labelling of an image; the same image may be labelled by up to 3 annotators. Every annotator batch is used as its own training sample, as the organisers suggest, and all filaments of a batch are merged into one binary mask. There is no validation split: all labelled data goes into training.

In [ ]:
def load_training_data(ann_path, img_dir):
    """Returns (samples, images, masks):
    samples: list of (image_id, file_name), one per annotator batch;
    images:  full-resolution grayscale images keyed by file_name (~3 GB in RAM);
    masks:   merged filament mask per image_id, kept as compact RLE and decoded on the fly."""
    with open(ann_path) as f:
        coco = json.load(f)
    anns = defaultdict(list)
    for ann in coco["annotations"]:
        anns[ann["image_id"]].append(ann)

    samples, images, masks = [], {}, {}
    for im in tqdm(coco["images"], desc="loading"):
        fn, h, w = im["file_name"], im["height"], im["width"]
        if fn not in images:
            images[fn] = cv2.imread(str(img_dir / fn), cv2.IMREAD_GRAYSCALE)
        polygons = [p for ann in anns[im["id"]] for p in ann["segmentation"]]
        masks[im["id"]] = (mask_utils.merge(mask_utils.frPyObjects(polygons, h, w)) if polygons
                           else mask_utils.encode(np.zeros((h, w), np.uint8, order="F")))
        samples.append((im["id"], fn))
    return samples, images, masks

NORM = A.Normalize(mean=(0.449,), std=(0.226,))  # ImageNet grey stats
train_tf = A.Compose([
    A.RandomCrop(CFG["crop_size"], CFG["crop_size"]),
    A.HorizontalFlip(), A.VerticalFlip(), A.RandomRotate90(),
    A.Affine(scale=(0.9, 1.1), rotate=(-15, 15), p=0.5),
    A.RandomBrightnessContrast(0.2, 0.2, p=0.5),
    A.RandomGamma(p=0.3),
    A.GaussianBlur(blur_limit=(3, 5), p=0.2),
    NORM, ToTensorV2(),
])
test_tf = A.Compose([NORM, ToTensorV2()])

class FilamentDataset(Dataset):
    """(augmented image crop, binary filament mask) pairs."""
    def __init__(self, samples, images, masks, tf):
        self.samples, self.images, self.masks, self.tf = samples, images, masks, tf
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, i):
        image_id, file_name = self.samples[i]
        mask = mask_utils.decode(self.masks[image_id]).astype(np.float32)
        out = self.tf(image=self.images[file_name], mask=mask)
        return out["image"], out["mask"].unsqueeze(0)

## 2. Model & training
`smp.Unet` with `in_channels=1` (smp adapts the pretrained first conv to grayscale) and one output channel (filament logit). Loss = BCE + Dice from `smp.losses`; AdamW with a one-cycle learning-rate schedule and mixed precision. Training runs for a fixed number of epochs and keeps the final weights (there is no validation set to pick an epoch with; the schedule ends at a near-zero learning rate).

In [ ]:
def build_model(pretrained):
    return smp.Unet(CFG["encoder"], encoder_weights="imagenet" if pretrained else None,
                    in_channels=1, classes=1).to(DEVICE)

def train(model, loader):
    bce = smp.losses.SoftBCEWithLogitsLoss()
    dice = smp.losses.DiceLoss("binary", from_logits=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=CFG["lr"],
                                                    total_steps=CFG["epochs"] * len(loader))
    scaler = torch.amp.GradScaler(enabled=DEVICE == "cuda")
    for epoch in range(CFG["epochs"]):
        model.train(); total = 0
        for x, y in tqdm(loader, leave=False, desc=f"epoch {epoch}"):
            x, y = x.to(DEVICE), y.to(DEVICE)
            with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
                logits = model(x)
                loss = bce(logits, y) + dice(logits, y)
            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.step(optimizer); scaler.update(); scheduler.step()
            total += loss.item()
        print(f"epoch {epoch:02d}  loss {total / len(loader):.4f}")

In [ ]:
model = build_model(pretrained=TRAIN)
if TRAIN:
    samples, images, masks = load_training_data(ANN_PATH, TRAIN_IMG_DIR)
    print(f"{len(images)} images, {len(samples)} annotated samples")
    n_gpus = max(1, torch.cuda.device_count())
    loader = DataLoader(FilamentDataset(samples, images, masks, train_tf), batch_size=CFG["batch_size"] * n_gpus,
                        shuffle=True, num_workers=CFG["num_workers"], drop_last=True, pin_memory=True,
                        persistent_workers=CFG["num_workers"] > 0)
    # With several GPUs (e.g. Kaggle "T4 x2") each batch is split across them; weights are shared with `model`.
    train(torch.nn.DataParallel(model) if n_gpus > 1 else model, loader)
    WEIGHTS.parent.mkdir(exist_ok=True)
    torch.save(model.state_dict(), WEIGHTS)
else:
    model.load_state_dict(torch.load(WEIGHTS, map_location=DEVICE))
model.eval();

## 3. Prediction & post-processing
* `predict`: average of 4 flipped predictions (TTA) on the full 2048×2048 image.
* `postprocess`: threshold → close small gaps in the outline and fill holes (annotations are smooth polygons without holes) → label connected components of a *dilated* mask so nearby fragments of one filament share an id, keeping labels only on the original pixels → drop components that are small or that the model is unsure about (mean probability below `min_conf`).

In [ ]:
def disk(r):
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * r + 1, 2 * r + 1))

def postprocess(prob, thr, merge_r, min_area, close_r, min_conf):
    """Probability map -> instance map (0 = background, k = k-th filament)."""
    mask = (prob > thr).astype(np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, disk(close_r))
    mask = ndimage.binary_fill_holes(mask).astype(np.uint8)
    _, labels = cv2.connectedComponents(cv2.dilate(mask, disk(merge_r)), connectivity=8)
    labels *= mask
    drop = [r.label for r in regionprops(labels, intensity_image=prob)
            if r.area < min_area or r.intensity_mean < min_conf]
    labels[np.isin(labels, drop)] = 0
    return labels

@torch.no_grad()
def predict(img_u8):
    """Flip-TTA filament probability map at the image's own resolution."""
    x = test_tf(image=img_u8)["image"][None].to(DEVICE)
    flips = [[], [-1], [-2], [-2, -1]]
    with torch.autocast(DEVICE, enabled=DEVICE == "cuda"):
        p = sum(torch.sigmoid(model(x.flip(f)).float()).flip(f) for f in flips) / len(flips)
    return p[0, 0].cpu().numpy()

def rle_counts(mask):
    return mask_utils.encode(np.asfortranarray(mask.astype(np.uint8)))["counts"].decode()

## 4. Test inference → `submission.csv`
One row per predicted filament; an image with no detections gets a single empty mask so every test image appears in the file.

In [ ]:
rows = []
for path in tqdm(sorted(TEST_IMG_DIR.glob("*.jpeg")), desc="test"):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    labels = postprocess(predict(img), **POST)
    ids = [k for k in np.unique(labels) if k != 0]
    for n, k in enumerate(ids, start=1):
        rows.append({"filament_id": f"{path.stem}_{n}", "segmentation_rle": rle_counts(labels == k)})
    if not ids:
        rows.append({"filament_id": f"{path.stem}_1", "segmentation_rle": rle_counts(np.zeros_like(img))})

sub = pd.DataFrame(rows)
sub.to_csv("submission.csv", index=False)
print(sub.shape, sub.filament_id.str.rsplit("_", n=1).str[0].nunique(), "images")
sub.head()

In [ ]:
# Sanity check on the last test image: input vs predicted filaments.
fig, ax = plt.subplots(1, 2, figsize=(14, 7))
ax[0].imshow(img, cmap="gray"); ax[0].set_title(path.name)
ax[1].imshow(labels, cmap="nipy_spectral", interpolation="nearest"); ax[1].set_title(f"{len(ids)} filaments")
for a in ax: a.axis("off")
plt.show()

## 5. Environment
Records the exact versions of the packages used in this run into `requirements.txt`.

In [ ]:
packages = ["torch", "torchvision", "timm", "segmentation-models-pytorch", "albumentations", "pycocotools",
            "opencv-python", "opencv-python-headless", "numpy", "pandas", "scipy", "scikit-image",
            "matplotlib", "tqdm"]
pins = []
for p in packages:
    try:
        pins.append(f"{p}=={version(p).split('+')[0]}")  # drop local tags like +cu121
    except PackageNotFoundError:
        pass
Path("requirements.txt").write_text("\n".join(pins) + "\n")
print("\n".join(pins))